# LLM Training Loop

This notebook now contains complete worked solutions for the Topic 5 exercises on **Training loop**.

Each section includes:
- a short explanation of the concept being tested
- direct answers to the written questions
- executable reference implementations
- assertions or sanity checks against expected behavior

The final section runs a compact end-to-end language-model training loop with checkpoint save/resume, so the notebook works as a practical Google Colab training demo.


In [ ]:
# Environment bootstrap for Google Colab and local notebooks.
# Run this cell first if the runtime is missing the assignment packages.

import sys
import subprocess

REQUIRED_PACKAGES = [
    "einops>=0.8",
    "einx>=0.4",
    "jaxtyping>=0.3",
    "numpy>=2.4",
    "psutil>=7",
    "pytest>=9.0",
    "pytest-timeout",
    "pypdf>=6.10.0",
    "regex>=2026.3.32",
    "tiktoken>=0.12.0",
    "torch~=2.11.0",
    "tqdm>=4.67",
    "wandb>=0.25",
    "ty>=0.0.26",
    "ruff>=0.15.8",
]

if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *REQUIRED_PACKAGES])
else:
    print("Local runtime detected; using the currently selected kernel environment.")


In [ ]:
# Shared imports and helper snippets for Topic 5 exercises.

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import math
import random
import tempfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

PROJECT_ROOT = Path.cwd()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Project root:", PROJECT_ROOT)
print("Torch version:", torch.__version__)
print("Device:", DEVICE)


def print_shape(name, tensor):
    print(f"{name}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}, device={tensor.device}")


def get_batch(dataset, batch_size: int, context_length: int, device: torch.device | str = "cpu"):
    data = torch.as_tensor(dataset, dtype=torch.long)
    if data.ndim != 1:
        raise ValueError("dataset must be a 1D token array")
    max_start_exclusive = data.numel() - context_length
    if max_start_exclusive <= 0:
        raise ValueError("dataset is too short for the requested context_length")

    starts = torch.randint(0, max_start_exclusive, (batch_size,))
    x = torch.stack([data[s : s + context_length] for s in starts.tolist()])
    y = torch.stack([data[s + 1 : s + 1 + context_length] for s in starts.tolist()])
    return x.to(device), y.to(device)


def save_checkpoint(path, model: nn.Module, optimizer: torch.optim.Optimizer, iteration: int, **extra_state):
    payload = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "iteration": iteration,
    }
    payload.update(extra_state)
    torch.save(payload, path)
    return payload


def load_checkpoint(path, model: nn.Module, optimizer: torch.optim.Optimizer, map_location="cpu"):
    payload = torch.load(path, map_location=map_location)
    model.load_state_dict(payload["model_state_dict"])
    optimizer.load_state_dict(payload["optimizer_state_dict"])
    return payload


def get_lr_cosine_schedule(it: int, *, max_lr: float, min_lr: float, warmup_iters: int, cosine_cycle_iters: int) -> float:
    if warmup_iters > 0 and it < warmup_iters:
        return max_lr * it / warmup_iters
    if it <= cosine_cycle_iters:
        span = max(1, cosine_cycle_iters - warmup_iters)
        progress = (it - warmup_iters) / span
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return min_lr + cosine * (max_lr - min_lr)
    return min_lr


## data_loader

**Assignment description**

- Implement Section 5.1 **Data Loader**.
- Match `tests/test_data.py::test_get_batch`.
- Sample random contiguous subsequences from a 1D token array.
- Return `x` and `y` where `y` is `x` shifted one token to the right.

**What this is testing**

This exercise checks whether you can turn a flat tokenized corpus into valid language-model training examples. A correct model still trains badly if the batching logic is off by one or samples invalid spans.

**Pseudocode / attack plan**

```text
dataset is a 1D array of token ids
max_start = len(dataset) - context_length - 1
sample batch_size random start indices from [0, max_start]
for each start index s:
    x_row = dataset[s : s + context_length]
    y_row = dataset[s + 1 : s + 1 + context_length]
stack rows into tensors of shape (batch_size, context_length)
move tensors to the requested device
return x, y
```

**Implementation notes**

- The start index must leave room for both the input span and the one-step-shifted target span.
- The target batch should satisfy `y == x + 1` in the toy `np.arange` test fixture.


In [ ]:
# data_loader completed solution

dataset = np.arange(20)
batch_size = 3
context_length = 5
x, y = get_batch(dataset, batch_size=batch_size, context_length=context_length, device="cpu")

print(x)
print(y)
print("shift check:", torch.equal(x + 1, y))
assert x.shape == (batch_size, context_length)
assert y.shape == (batch_size, context_length)
assert torch.equal(x + 1, y)

for _ in range(20):
    xb, yb = get_batch(np.arange(11), batch_size=4, context_length=5)
    assert xb.max().item() <= 9
    assert yb.max().item() <= 10
    assert torch.equal(xb + 1, yb)

answer_upper_bound = (
    "The upper bound for starting indices is exclusive because the final valid start still needs room for both the input span and the shifted target span. "
    "If you sampled one step past that point, the target slice would be too short."
)
answer_shapes = (
    "x and y must have identical shapes because the model produces one next-token prediction per input position, so each logit row must align with exactly one target token."
)

print(answer_upper_bound)
print(answer_shapes)


## checkpointing

**Assignment description**

- Implement Section 5.2 **Checkpointing**.
- Match `tests/test_serialization.py::test_checkpointing`.
- Save model state, optimizer state, and iteration count.
- Restore all of them correctly into a fresh model and optimizer.

**What this is testing**

This exercise checks whether you can pause and resume training without losing progress. That means parameters, optimizer moments, and step count all need to survive serialization.

**Pseudocode / attack plan**

```text
to save:
    package a dictionary with:
        model_state_dict
        optimizer_state_dict
        iteration
    torch.save(package, destination)

to load:
    package = torch.load(source)
    model.load_state_dict(package['model_state_dict'])
    optimizer.load_state_dict(package['optimizer_state_dict'])
    return package['iteration']
```

**Implementation notes**

- Saving only model weights is not enough if you want a true training resume.
- Optimizer state matters especially for AdamW because it stores running moments.


In [ ]:
# checkpointing completed solution

class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4, 2)

    def forward(self, x):
        return self.fc(x)


model = TinyNet()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
iteration = 7

with torch.no_grad():
    model.fc.weight.fill_(0.25)
    model.fc.bias.fill_(-0.5)

sample_x = torch.randn(3, 4)
loss = model(sample_x).pow(2).mean()
loss.backward()
optimizer.step()

ckpt_path = Path(tempfile.gettempdir()) / "topic5_checkpoint_demo.pt"
save_checkpoint(ckpt_path, model, optimizer, iteration, note="demo")

fresh_model = TinyNet()
fresh_optimizer = torch.optim.AdamW(fresh_model.parameters(), lr=1e-3)
loaded = load_checkpoint(ckpt_path, fresh_model, fresh_optimizer)

print("saved to:", ckpt_path)
print("keys:", sorted(loaded.keys()))
print("iteration:", loaded["iteration"])
assert loaded["iteration"] == iteration

for key, value in model.state_dict().items():
    assert torch.allclose(value, fresh_model.state_dict()[key])

for original_group, loaded_group in zip(optimizer.state_dict()["param_groups"], fresh_optimizer.state_dict()["param_groups"]):
    assert original_group["lr"] == loaded_group["lr"]

answer_optimizer = (
    "Optimizer state must be restored with model state because adaptive optimizers track momentum and second-moment history; without that state, resumed training no longer follows the same trajectory."
)
answer_iteration = (
    "The iteration count belongs in the checkpoint so logging, learning-rate scheduling, checkpoint cadence, and resume semantics all continue from the correct training step instead of restarting from zero."
)

print(answer_optimizer)
print(answer_iteration)


## training_loop

**Assignment description**

- Study Section 5.3 **Training loop**.
- Combine data loading, forward pass, loss computation, backward pass, gradient clipping, optimizer step, LR scheduling, and checkpointing.
- Write the full per-iteration training pipeline in the right order.

**What this is testing**

This topic checks whether you can orchestrate the whole system. Most practical bugs in training come from ordering mistakes: clipping at the wrong time, forgetting to zero gradients, saving incomplete checkpoints, or logging the wrong step.

**Pseudocode / attack plan**

```text
initialize model, optimizer, scheduler, and dataset
for iteration in range(max_iters):
    model.train()
    x, y = get_batch(...)
    optimizer.zero_grad()
    logits = model(x)
    loss = cross_entropy(logits reshaped appropriately, y reshaped appropriately)
    loss.backward()
    gradient_clipping(model.parameters(), max_norm)
    set optimizer learning rate from scheduler if needed
    optimizer.step()
    log metrics
    periodically save checkpoint
```

**Writeup guidance**

- Separate the conceptual order from framework-specific syntax.
- Be explicit about where training mode, zeroing gradients, and checkpoint intervals belong.


In [ ]:
# training_loop completed solution

class TinyLM(nn.Module):
    def __init__(self, vocab_size=32, d_model=16):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.proj = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids):
        x = self.emb(token_ids)
        return self.proj(x)


dataset = np.arange(200) % 32
model = TinyLM().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

batch_size = 4
context_length = 8
max_iters = 12
max_lr = 1e-3
min_lr = 1e-4
warmup_iters = 3
max_grad_norm = 1.0
checkpoint_path = Path(tempfile.gettempdir()) / "topic5_training_loop_demo.pt"

history = []
for iteration in range(max_iters):
    model.train()
    lr = get_lr_cosine_schedule(
        iteration,
        max_lr=max_lr,
        min_lr=min_lr,
        warmup_iters=warmup_iters,
        cosine_cycle_iters=max_iters,
    )
    for group in optimizer.param_groups:
        group["lr"] = lr

    x, y = get_batch(dataset, batch_size=batch_size, context_length=context_length, device=DEVICE)
    optimizer.zero_grad(set_to_none=True)
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
    optimizer.step()

    record = {
        "iteration": iteration,
        "lr": lr,
        "loss": float(loss.item()),
        "grad_norm": float(grad_norm),
    }
    history.append(record)

    if iteration % 4 == 0 or iteration == max_iters - 1:
        save_checkpoint(checkpoint_path, model, optimizer, iteration)
        print(record)

print_shape("x", x)
print_shape("logits", logits)
print("final loss:", history[-1]["loss"])
print("checkpoint exists:", checkpoint_path.exists())
assert checkpoint_path.exists()

answer_flatten = (
    "Logits and targets are flattened because cross-entropy expects a 2D tensor of class scores and a 1D tensor of class indices. "
    "Flattening converts (batch, seq, vocab) and (batch, seq) into one row per token prediction."
)
answer_order = (
    "The correct per-step order is: set train mode, fetch batch, zero grads, forward pass, compute loss, backward pass, clip gradients, apply optimizer step, log metrics, and checkpoint at the configured interval."
)

print(answer_flatten)
print(answer_order)


## end_to_end_training_resume_demo

The earlier cells solve the exercises independently. This final section uses the same helpers in one compact run that:
- builds a tiny character-level corpus
- trains a small causal LM for a few steps
- saves a checkpoint
- restores into a fresh model and optimizer
- continues training and generates a short sample

That gives you a minimal Colab-ready reference for how the full training loop fits together in practice.


In [ ]:
# end_to_end_training_resume_demo

corpus = "\n".join(
    [
        "training loops coordinate data, optimization, and checkpointing.",
        "language model batches are contiguous spans from a token stream.",
        "resuming from checkpoints should preserve optimizer state and step count.",
    ]
    * 120
)
chars = sorted(set(corpus))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda ids: "".join(itos[i] for i in ids)

all_tokens = np.array(encode(corpus), dtype=np.int64)
split = int(0.9 * len(all_tokens))
train_tokens = all_tokens[:split]
val_tokens = all_tokens[split:]


@dataclass
class TinyLoopConfig:
    vocab_size: int
    block_size: int = 40
    d_model: int = 48


class TinyCausalLM(nn.Module):
    def __init__(self, config: TinyLoopConfig):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.position_embedding = nn.Embedding(config.block_size, config.d_model)
        self.ln = nn.LayerNorm(config.d_model)
        self.mlp = nn.Sequential(
            nn.Linear(config.d_model, 2 * config.d_model),
            nn.GELU(),
            nn.Linear(2 * config.d_model, config.d_model),
        )
        self.head = nn.Linear(config.d_model, config.vocab_size, bias=False)

    def forward(self, idx, targets=None):
        _, seq_len = idx.shape
        positions = torch.arange(seq_len, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)[None, :, :]
        x = x + self.mlp(self.ln(x))
        logits = self.head(self.ln(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


def estimate_loss(model, split_tokens, *, eval_batches=6, batch_size=12, context_length=40):
    model.eval()
    losses = []
    with torch.no_grad():
        for _ in range(eval_batches):
            xb, yb = get_batch(split_tokens, batch_size=batch_size, context_length=context_length, device=DEVICE)
            _, loss = model(xb, yb)
            losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)


config = TinyLoopConfig(vocab_size=len(chars))
model = TinyCausalLM(config).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=0.01)
checkpoint_path = Path(tempfile.gettempdir()) / "topic5_resume_demo.pt"

train_steps_before_save = 18
train_steps_after_resume = 10
batch_size = 12
context_length = config.block_size
max_lr = 2e-3
min_lr = 2e-4
warmup_iters = 4
max_grad_norm = 1.0

initial_loss = estimate_loss(model, train_tokens, batch_size=batch_size, context_length=context_length)
print(f"initial train loss: {initial_loss:.4f}")

for iteration in range(train_steps_before_save):
    lr = get_lr_cosine_schedule(
        iteration,
        max_lr=max_lr,
        min_lr=min_lr,
        warmup_iters=warmup_iters,
        cosine_cycle_iters=train_steps_before_save + train_steps_after_resume,
    )
    for group in optimizer.param_groups:
        group["lr"] = lr
    xb, yb = get_batch(train_tokens, batch_size=batch_size, context_length=context_length, device=DEVICE)
    optimizer.zero_grad(set_to_none=True)
    _, loss = model(xb, yb)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
    optimizer.step()

save_checkpoint(checkpoint_path, model, optimizer, train_steps_before_save, vocab=chars)
print("checkpoint saved to:", checkpoint_path)

restored_model = TinyCausalLM(config).to(DEVICE)
restored_optimizer = torch.optim.AdamW(restored_model.parameters(), lr=2e-3, weight_decay=0.01)
loaded = load_checkpoint(checkpoint_path, restored_model, restored_optimizer, map_location=DEVICE)
start_iteration = loaded["iteration"]
print("resumed iteration:", start_iteration)

for iteration in range(start_iteration, start_iteration + train_steps_after_resume):
    lr = get_lr_cosine_schedule(
        iteration,
        max_lr=max_lr,
        min_lr=min_lr,
        warmup_iters=warmup_iters,
        cosine_cycle_iters=train_steps_before_save + train_steps_after_resume,
    )
    for group in restored_optimizer.param_groups:
        group["lr"] = lr
    xb, yb = get_batch(train_tokens, batch_size=batch_size, context_length=context_length, device=DEVICE)
    restored_optimizer.zero_grad(set_to_none=True)
    _, loss = restored_model(xb, yb)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(restored_model.parameters(), max_norm=max_grad_norm)
    restored_optimizer.step()

final_train_loss = estimate_loss(restored_model, train_tokens, batch_size=batch_size, context_length=context_length)
final_val_loss = estimate_loss(restored_model, val_tokens, batch_size=batch_size, context_length=context_length)
print(f"final train loss: {final_train_loss:.4f}")
print(f"final val loss:   {final_val_loss:.4f}")
assert final_train_loss < initial_loss


@torch.no_grad()
def generate(model, prompt: str, max_new_tokens: int = 100, temperature: float = 0.9):
    model.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=DEVICE)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -config.block_size :]
        logits, _ = model(idx_cond)
        next_token_logits = logits[:, -1, :] / temperature
        probs = torch.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_token], dim=1)
    return decode(idx[0].tolist())


sample = generate(restored_model, prompt="training loops ")
print("generated sample:\n")
print(sample)
